# Azure MLOps Complete Workflow

End-to-end guide for deploying the MLOps platform on Microsoft Azure.

## Workflow

| Step | Folder | Description |
|------|--------|-------------|
| 1 | `01-azure-blob-storage-setup/` | Create storage account + blob container |
| 2 | `02-model-training/` | Train TinyBERT + ViT models, push to Azure Blob |
| 3 | `03-azure-vm-setup/` | Provision Azure VM |
| 4 | `04-local-app-development/` | FastAPI + Streamlit local dev |
| 5 | `05-deploy-streamlit-vm/` | Deploy Streamlit to Azure VM |
| 6 | `06-dockerize-app-and-fastapi/` | Containerize FastAPI + Nginx |
| 7 | `07-deploy-azure-container-apps/` | Push image to ACR, deploy Container App |

**Python 3.11.8 throughout.**

In [ ]:
# Install all Azure SDK dependencies
!pip install \
    "azure-identity>=1.15.0,<2.0.0" \
    "azure-core>=1.30.0,<2.0.0" \
    "azure-mgmt-resource>=23.0.0,<24.0.0" \
    "azure-mgmt-storage>=24.0.0,<25.0.0" \
    "azure-mgmt-compute>=30.0.0,<31.0.0" \
    "azure-mgmt-network>=25.0.0,<26.0.0" \
    "azure-mgmt-authorization>=4.0.0,<5.0.0" \
    "azure-mgmt-containerregistry>=10.0.0,<11.0.0" \
    "azure-mgmt-containerinstance>=10.0.0,<11.0.0" \
    "azure-storage-blob>=12.19.0,<13.0.0" \
    "python-dotenv>=1.0.0,<2.0.0" \
    "requests"

## Authentication

**Option A — Recommended for local development:**
```bash
az login
```
`DefaultAzureCredential` will pick up your `az login` session automatically.

**Option B — Service Principal (CI/CD or automation):**
```bash
export AZURE_TENANT_ID=your-tenant-id
export AZURE_CLIENT_ID=your-client-id
export AZURE_CLIENT_SECRET=your-client-secret
```
Add these to your `.env` file (see `01-azure-blob-storage-setup/.env.example`).
`DefaultAzureCredential` checks these environment variables before falling back to `az login`.

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.mgmt.resource import ResourceManagementClient
from azure.mgmt.storage import StorageManagementClient
from azure.mgmt.compute import ComputeManagementClient
from azure.mgmt.network import NetworkManagementClient
from azure.mgmt.authorization import AuthorizationManagementClient
from azure.mgmt.containerregistry import ContainerRegistryManagementClient

load_dotenv()

SUBSCRIPTION_ID = os.environ['AZURE_SUBSCRIPTION_ID']
RESOURCE_GROUP = os.environ.get('AZURE_RESOURCE_GROUP', 'mlops-rg')
LOCATION = os.environ.get('AZURE_LOCATION', 'eastus')
STORAGE_ACCOUNT_NAME = os.environ['AZURE_STORAGE_ACCOUNT_NAME']
ACR_NAME = os.environ.get('AZURE_ACR_NAME', 'mlopsacr')
VM_NAME = os.environ.get('AZURE_VM_NAME', 'mlops-vm')

credential = DefaultAzureCredential()
resource_client = ResourceManagementClient(credential, SUBSCRIPTION_ID)
storage_mgmt = StorageManagementClient(credential, SUBSCRIPTION_ID)
compute_client = ComputeManagementClient(credential, SUBSCRIPTION_ID)
network_client = NetworkManagementClient(credential, SUBSCRIPTION_ID)
auth_client = AuthorizationManagementClient(credential, SUBSCRIPTION_ID)
acr_client = ContainerRegistryManagementClient(credential, SUBSCRIPTION_ID)

print(f'Subscription:   {SUBSCRIPTION_ID}')
print(f'Resource Group: {RESOURCE_GROUP}')
print(f'Location:       {LOCATION}')
print(f'Storage Acct:   {STORAGE_ACCOUNT_NAME}')
print('All SDK clients initialized.')

In [ ]:
# Step 1: Create Resource Group
rg = resource_client.resource_groups.create_or_update(
    RESOURCE_GROUP,
    {'location': LOCATION}
)
print(f'Resource Group: {rg.name} ({rg.location}) — {rg.properties.provisioning_state}')

## Azure RBAC — Least Privilege

Azure uses **Role-Based Access Control (RBAC)** with built-in role definitions.
Roles are bound to a **principal** (Service Principal / App Registration) at a **scope** (resource / resource group / subscription).

| Built-in Role | GUID | Scope | Purpose |
|---------------|------|-------|---------|
| Storage Blob Data Contributor | `ba92f5b4-...` | Storage account | Read + write model blobs |
| Virtual Machine Contributor | `9980e02c-...` | Resource group | Manage VM |
| AcrPush | `8311e382-...` | ACR resource | Push Docker images |
| AcrPull | `7f951dda-...` | ACR resource | Pull Docker images |

These GUIDs are **stable across all Azure tenants** — they never change.

**Key difference from GCP:** Azure scopes roles to resource IDs via ARM paths like
`/subscriptions/{sub}/resourceGroups/{rg}/providers/Microsoft.Storage/storageAccounts/{name}`
rather than calling a resource-specific `set_iam_policy()` method.

In [ ]:
# Get Service Principal Object ID
# NOTE: Object ID != Client ID (app registration ID)
# The role assignment requires the Object ID of the service principal
import subprocess, json

CLIENT_ID = os.environ.get('AZURE_CLIENT_ID', '')
if CLIENT_ID:
    result = subprocess.run(
        ['az', 'ad', 'sp', 'show', '--id', CLIENT_ID, '--query', 'id', '-o', 'tsv'],
        capture_output=True, text=True
    )
    SP_OBJECT_ID = result.stdout.strip()
    print(f'Service Principal Object ID: {SP_OBJECT_ID}')
else:
    SP_OBJECT_ID = None
    print('AZURE_CLIENT_ID not set — skipping SP lookup. Set it to assign RBAC roles.')

In [ ]:
# Assign RBAC roles (resource-scoped)
from uuid import uuid4
from azure.mgmt.authorization.models import RoleAssignmentCreateParameters

# Stable built-in role GUIDs (valid in all Azure tenants)
ROLE_GUIDS = {
    'Storage Blob Data Contributor': 'ba92f5b4-2d11-453d-a403-e96b0029c9fe',
    'Virtual Machine Contributor':   '9980e02c-c2be-4d73-94e8-173b1dc7cf3c',
    'AcrPush':                       '8311e382-0749-4cb8-b61a-304f252e4c2e',
    'AcrPull':                       '7f951dda-4ed3-4680-a7ca-43fe172d538d',
}

def assign_role(scope, role_name, principal_id):
    guid = ROLE_GUIDS[role_name]
    role_def_id = f'/subscriptions/{SUBSCRIPTION_ID}/providers/Microsoft.Authorization/roleDefinitions/{guid}'
    try:
        auth_client.role_assignments.create(
            scope=scope,
            role_assignment_name=str(uuid4()),
            parameters=RoleAssignmentCreateParameters(
                role_definition_id=role_def_id,
                principal_id=principal_id,
                principal_type='ServicePrincipal',
            )
        )
        print(f'  Assigned: {role_name}')
    except Exception as e:
        if 'RoleAssignmentExists' in str(e):
            print(f'  Already assigned: {role_name}')
        else:
            raise

if SP_OBJECT_ID:
    # Storage Blob Data Contributor — scoped to storage account (not subscription)
    storage_scope = (
        f'/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}'
        f'/providers/Microsoft.Storage/storageAccounts/{STORAGE_ACCOUNT_NAME}'
    )
    assign_role(storage_scope, 'Storage Blob Data Contributor', SP_OBJECT_ID)

    # VM Contributor — scoped to resource group
    rg_scope = f'/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}'
    assign_role(rg_scope, 'Virtual Machine Contributor', SP_OBJECT_ID)

    print('RBAC assignments complete (ACR roles will be assigned after ACR is created).')
else:
    print('Skipped RBAC assignment — set AZURE_CLIENT_ID to enable.')

In [ ]:
# Step 1: Create Storage Account + Blob Container
from azure.mgmt.storage.models import StorageAccountCreateParameters, Sku, Kind
from azure.storage.blob import BlobServiceClient

print(f'Creating storage account {STORAGE_ACCOUNT_NAME!r}...')
op = storage_mgmt.storage_accounts.begin_create(
    RESOURCE_GROUP,
    STORAGE_ACCOUNT_NAME,
    StorageAccountCreateParameters(
        sku=Sku(name='Standard_LRS'),
        kind=Kind.STORAGE_V2,
        location=LOCATION,
    )
)
storage_account = op.result()
print(f'Storage account: {storage_account.name} ({storage_account.provisioning_state})')

# Get connection string
keys = storage_mgmt.storage_accounts.list_keys(RESOURCE_GROUP, STORAGE_ACCOUNT_NAME)
CONN_STR = (
    f'DefaultEndpointsProtocol=https;'
    f'AccountName={STORAGE_ACCOUNT_NAME};'
    f'AccountKey={keys.keys[0].value};'
    f'EndpointSuffix=core.windows.net'
)

# Create blob container
blob_service = BlobServiceClient.from_connection_string(CONN_STR)
container_client = blob_service.get_container_client('models')
try:
    container_client.create_container()
    print('Container "models" created.')
except Exception:
    print('Container "models" already exists.')

In [ ]:
# Blob upload/download test
test_name = '_test/hello.txt'
container_client.upload_blob(test_name, b'Hello from MLOps!', overwrite=True)
print('Upload OK')

content = container_client.download_blob(test_name).readall()
print(f'Download OK: {content!r}')

container_client.delete_blob(test_name)
print('Test blob cleaned up')

In [ ]:
# Step 3: Create Azure VM with NSG
from azure.mgmt.network.models import (
    VirtualNetwork, AddressSpace, Subnet,
    PublicIPAddress, PublicIPAddressSku,
    NetworkInterface, NetworkInterfaceIPConfiguration,
    NetworkSecurityGroup, SecurityRule,
)
from azure.mgmt.compute.models import (
    VirtualMachine, HardwareProfile, StorageProfile,
    OSDisk, ImageReference, OSProfile, LinuxConfiguration,
    SshConfiguration, SshPublicKey, NetworkProfile, NetworkInterfaceReference,
)

print('Creating network resources...')

# VNet
vnet_op = network_client.virtual_networks.begin_create_or_update(
    RESOURCE_GROUP, 'mlops-vnet',
    VirtualNetwork(location=LOCATION, address_space=AddressSpace(address_prefixes=['10.0.0.0/16']))
)
vnet_op.result()

# Subnet
subnet_op = network_client.subnets.begin_create_or_update(
    RESOURCE_GROUP, 'mlops-vnet', 'mlops-subnet',
    Subnet(address_prefix='10.0.1.0/24')
)
subnet = subnet_op.result()

# Public IP
pip_op = network_client.public_ip_addresses.begin_create_or_update(
    RESOURCE_GROUP, 'mlops-pip',
    PublicIPAddress(
        location=LOCATION,
        sku=PublicIPAddressSku(name='Standard'),
        public_ip_allocation_method='Static',
    )
)
pip = pip_op.result()

# NSG with SSH rule
nsg_op = network_client.network_security_groups.begin_create_or_update(
    RESOURCE_GROUP, 'mlops-nsg',
    NetworkSecurityGroup(
        location=LOCATION,
        security_rules=[
            SecurityRule(
                name='allow-ssh', protocol='Tcp', priority=1000, direction='Inbound',
                access='Allow', source_port_range='*', destination_port_range='22',
                source_address_prefix='*', destination_address_prefix='*',
            )
        ]
    )
)
nsg = nsg_op.result()

# NIC
nic_op = network_client.network_interfaces.begin_create_or_update(
    RESOURCE_GROUP, 'mlops-nic',
    NetworkInterface(
        location=LOCATION,
        network_security_group=nsg,
        ip_configurations=[
            NetworkInterfaceIPConfiguration(
                name='ipconfig1', subnet=subnet, public_ip_address=pip,
                private_ip_allocation_method='Dynamic',
            )
        ]
    )
)
nic = nic_op.result()
print(f'Network resources created. Public IP: {pip.ip_address}')

In [ ]:
# Create VM
SSH_PUBLIC_KEY_PATH = os.path.expanduser('~/.ssh/id_rsa.pub')
with open(SSH_PUBLIC_KEY_PATH) as f:
    ssh_key_data = f.read().strip()

vm_op = compute_client.virtual_machines.begin_create_or_update(
    RESOURCE_GROUP, VM_NAME,
    VirtualMachine(
        location=LOCATION,
        hardware_profile=HardwareProfile(vm_size='Standard_D2s_v3'),
        storage_profile=StorageProfile(
            image_reference=ImageReference(
                publisher='Canonical', offer='0001-com-ubuntu-server-jammy',
                sku='22_04-lts', version='latest',
            ),
            os_disk=OSDisk(create_option='FromImage', disk_size_gb=64),
        ),
        os_profile=OSProfile(
            computer_name=VM_NAME, admin_username='azureuser',
            linux_configuration=LinuxConfiguration(
                disable_password_authentication=True,
                ssh=SshConfiguration(public_keys=[
                    SshPublicKey(
                        path='/home/azureuser/.ssh/authorized_keys',
                        key_data=ssh_key_data,
                    )
                ])
            )
        ),
        network_profile=NetworkProfile(
            network_interfaces=[NetworkInterfaceReference(id=nic.id, primary=True)]
        ),
    )
)
vm = vm_op.result()
print(f'VM created: {vm.name} — SSH: azureuser@{pip.ip_address}')

## Steps 4–5: Local Development & Streamlit

```bash
# Step 4 — FastAPI locally
cd infrastructure/azure/04-local-app-development/fastapi
pip install -r requirements.txt
uvicorn main:app --reload --port 5000

# Step 4 — Streamlit locally (in a second terminal)
cd infrastructure/azure/04-local-app-development/streamlit
streamlit run app.py

# Step 5 — Deploy Streamlit to Azure VM
# See 05-deploy-streamlit-vm/vm-streamlit-deploy.ipynb
```

In [ ]:
# Step 6: Create ACR and print docker commands
from azure.mgmt.containerregistry.models import Registry, Sku

print(f'Creating ACR {ACR_NAME!r}...')
op = acr_client.registries.begin_create(
    RESOURCE_GROUP, ACR_NAME,
    Registry(location=LOCATION, sku=Sku(name='Basic'), admin_user_enabled=True)
)
acr = op.result()
print(f'ACR login server: {acr.login_server}')

IMAGE_URI = f'{acr.login_server}/mlops-api:latest'

if SP_OBJECT_ID:
    acr_scope = (
        f'/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}'
        f'/providers/Microsoft.ContainerRegistry/registries/{ACR_NAME}'
    )
    assign_role(acr_scope, 'AcrPush', SP_OBJECT_ID)
    assign_role(acr_scope, 'AcrPull', SP_OBJECT_ID)

print()
print('Docker commands (run in terminal):')
print(f'  az acr login --name {ACR_NAME}')
print(f'  docker build -t mlops-api infrastructure/azure/06-dockerize-app-and-fastapi/')
print(f'  docker tag mlops-api {IMAGE_URI}')
print(f'  docker push {IMAGE_URI}')

In [ ]:
# Step 7: Deploy to Azure Container Apps
from azure.mgmt.appcontainers import ContainerAppsAPIClient
from azure.mgmt.appcontainers.models import (
    ManagedEnvironment, ContainerApp, Configuration,
    Ingress, RegistryCredentials, Template, Container, Secret, EnvironmentVar,
)

aca_client = ContainerAppsAPIClient(credential, SUBSCRIPTION_ID)

# Get ACR credentials
creds = acr_client.registries.list_credentials(RESOURCE_GROUP, ACR_NAME)
ACR_USERNAME = creds.username
ACR_PASSWORD = creds.passwords[0].value

# Create Container Apps environment
print('Creating Container Apps environment...')
env_op = aca_client.managed_environments.begin_create_or_update(
    RESOURCE_GROUP, 'mlops-env', ManagedEnvironment(location=LOCATION)
)
env = env_op.result()
print(f'Environment: {env.name}')

# Deploy Container App
print('Deploying Container App...')
app_op = aca_client.container_apps.begin_create_or_update(
    RESOURCE_GROUP, 'mlops-api',
    ContainerApp(
        location=LOCATION,
        managed_environment_id=env.id,
        configuration=Configuration(
            ingress=Ingress(external=True, target_port=5000),
            registries=[RegistryCredentials(
                server=acr.login_server,
                username=ACR_USERNAME,
                password_secret_ref='acr-password',
            )],
            secrets=[
                Secret(name='acr-password', value=ACR_PASSWORD),
                Secret(name='storage-connection', value=CONN_STR),
            ],
        ),
        template=Template(containers=[Container(
            name='mlops-api',
            image=IMAGE_URI,
            env=[EnvironmentVar(
                name='AZURE_STORAGE_CONNECTION_STRING',
                secret_ref='storage-connection',
            )],
            resources={'cpu': 1.0, 'memory': '2Gi'},
        )]),
    )
)
app = app_op.result()
APP_URL = f'https://{app.configuration.ingress.fqdn}'
print(f'Container App URL: {APP_URL}')

In [ ]:
# Verify endpoints + teardown
import requests, time

time.sleep(10)
resp = requests.get(APP_URL, timeout=30)
print(f'Health: {resp.status_code} — {resp.json()}')

resp = requests.post(
    f'{APP_URL}/api/v1/pose_classifier',
    json={'url': ['https://images.pexels.com/photos/1755385/pexels-photo-1755385.jpeg']},
    timeout=60,
)
print(f'Pose classifier: {resp.status_code}')
print(resp.json())

In [ ]:
# Teardown — delete entire resource group (atomically removes all resources)
# Uncomment to execute:
# op = resource_client.resource_groups.begin_delete(RESOURCE_GROUP)
# op.result()
# print(f'Resource group {RESOURCE_GROUP!r} deleted.')
print('Teardown skipped. Uncomment above to delete the entire resource group.')

## Workflow Summary

| Step | Folder | What Was Done |
|------|--------|---------------|
| 1 | `01-azure-blob-storage-setup/` | Created storage account + blob container |
| 2 | `02-model-training/` | Trained models, pushed to Azure Blob |
| 3 | `03-azure-vm-setup/` | Created Azure VM with NSG |
| 4 | `04-local-app-development/` | Developed FastAPI + Streamlit locally |
| 5 | `05-deploy-streamlit-vm/` | Deployed Streamlit to Azure VM |
| 6 | `06-dockerize-app-and-fastapi/` | Containerized app, pushed to ACR |
| 7 | `07-deploy-azure-container-apps/` | Deployed serverless Container App |

## IAM Comparison: Azure vs AWS

| Aspect | Azure | AWS |
|--------|-------|-----|
| Identity type | Service Principal (App Registration) | IAM Role + Instance Profile |
| Storage scope | `role_assignments.create()` scoped to storage account ARM path | Inline policy with specific S3 bucket ARN |
| Registry scope | `AcrPush`/`AcrPull` scoped to ACR resource | ECR resource-based policy |
| Compute scope | VM Contributor scoped to resource group | Instance Profile attached to EC2 |
| Serverless scope | Container App uses managed identity or env var secrets | ECS task execution role with ECR pull permission |
| Teardown | Delete resource group removes all resources atomically | Manual deletion per service |